# Shakespeare's Characters

The goal of this notebook is to train a simple LSTM model **to mimic the dialogue of Shakespearean characters.** We will train the model on a large corpus of text from Shakespeare's plays, teaching it **to predict the next character in a sequence.** For example, given the input 'to be or no', the model should predict 't' as the most likely following character. The input window then slides forward by one position—now reading 'o be or not'—to predict the next token. This sliding window approach allows the model to generate text character-by-character continuously.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np

In [ ]:
# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Data Loading

In [ ]:
# Download and prepare data
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
text = open('input.txt', 'r').read()

--2026-03-20 16:50:27--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.007s  

2026-03-20 16:50:27 (160 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
# Create character-level vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

# Encode the entire text into integers
data = [stoi[c] for c in text]

In [ ]:
# Sliding Window Dataset
# Instead of sentences, we slice the giant text into overlapping sequences

class CharDataset(Dataset):
    def __init__(self, data, seq_len=100):
        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        # x is a sequence of seq_len characters
        # y is the single character that comes right after it
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + self.seq_len]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

In [ ]:
# Parameters
seq_len = 100
batch_size = 128

# Create the dataset
dataset = CharDataset(data, seq_len)

# Create the data loader
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

## Model Definition

In [ ]:
class ShakespeareLSTM(nn.Module):
    def __init__(self, vocab_size, embed_size=128, hidden_size=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        # x: [batch, seq_len]
        emb = self.embedding(x)
        out, _ = self.lstm(emb)
        # Use the last hidden state for prediction
        logits = self.fc(out[:, -1, :])
        return logits

In [ ]:
# Instantiate the model
model = ShakespeareLSTM(vocab_size).to(device)

##Training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)

In [ ]:
epochs = 5
model.train()

for epoch in range(epochs):
    total_loss = 0
    for i, (x, y) in enumerate(loader):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if i % 500 == 0:
            print(f"Epoch {epoch+1}, Batch {i}, Loss: {loss.item():.4f}")

    print(f"--- Epoch {epoch+1} Avg Loss: {total_loss/len(loader):.4f} ---")

Epoch 1, Batch 0, Loss: 4.1781
Epoch 1, Batch 500, Loss: 1.9559
Epoch 1, Batch 1000, Loss: 1.7515
Epoch 1, Batch 1500, Loss: 1.9214
Epoch 1, Batch 2000, Loss: 1.6626
Epoch 1, Batch 2500, Loss: 1.2778
Epoch 1, Batch 3000, Loss: 1.6483
Epoch 1, Batch 3500, Loss: 1.6897
Epoch 1, Batch 4000, Loss: 1.7375
Epoch 1, Batch 4500, Loss: 1.3790
Epoch 1, Batch 5000, Loss: 1.6283
Epoch 1, Batch 5500, Loss: 1.5071
Epoch 1, Batch 6000, Loss: 1.5991
Epoch 1, Batch 6500, Loss: 1.5122
Epoch 1, Batch 7000, Loss: 1.6376
Epoch 1, Batch 7500, Loss: 1.5705
Epoch 1, Batch 8000, Loss: 1.3527
Epoch 1, Batch 8500, Loss: 1.4082
--- Epoch 1 Avg Loss: 1.6189 ---
Epoch 2, Batch 0, Loss: 1.3930
Epoch 2, Batch 500, Loss: 1.6023
Epoch 2, Batch 1000, Loss: 1.4097
Epoch 2, Batch 1500, Loss: 1.5166
Epoch 2, Batch 2000, Loss: 1.3376
Epoch 2, Batch 2500, Loss: 1.4018
Epoch 2, Batch 3000, Loss: 1.4078
Epoch 2, Batch 3500, Loss: 1.3025
Epoch 2, Batch 4000, Loss: 1.1950
Epoch 2, Batch 4500, Loss: 1.4787
Epoch 2, Batch 5000, Lo

## Validation

In [ ]:
def generate_text(model, start_str="ROMEO: ", length=200, temperature=0.8):
    model.eval()
    generated = start_str

    for _ in range(length):
        # Format input string to length seq_len
        input_tokens = [stoi[c] for c in generated[-seq_len:]]
        x = torch.tensor(input_tokens).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(x)

            # Apply temperature and sample (instead of argmax)
            # Higher temp = more "creative"/chaotic text
            probs = torch.softmax(logits / temperature, dim=-1)
            pred_idx = torch.multinomial(probs, num_samples=1).item()

            generated += itos[pred_idx]

    return generated

In [ ]:
print("\nGenerated Shakespeare:\n")
print(generate_text(model, start_str="KING: ", length=300))


Generated Shakespeare:

KING: I saw he
a kin persemble and remembers round in again.

LUCIO:
Come, good lord, I know he come away.

WARWICK:
Ay, in not be suppecting for my success,
I'll be a prince, the better tears are!

CATESBY:
I will make utters, fields upon the sweets in your prison.

BAPTISTA:
Of well, boy, let me the tim
